In [ ]:
from datetime import datetime
import pandas as pd
import plotly.graph_objects as go
import pytz

In [ ]:
# Load the dataset
apps_df = pd.read_csv('Play Store Data.csv')
reviews_df = pd.read_csv('User Reviews.csv')
apps_df.head(3)

In [ ]:
# Drop missing values
apps_df = apps_df.dropna(subset=['Category', 'Installs']).copy()

# Clean Installs: remove '+' and ',' then convert to integer
apps_df['Installs'] = apps_df['Installs'].astype(str).str.replace('+', '').str.replace(',', '')
apps_df = apps_df[apps_df['Installs'].str.isnumeric()]
apps_df['Installs'] = apps_df['Installs'].astype(int)

# Group by Category and filter out categories starting with A, C, G, S
category_df = apps_df.groupby('Category')['Installs'].sum().reset_index()
filtered_df = category_df[~category_df['Category'].str.startswith(('A', 'C', 'G', 'S'))]

# Get Top 5 categories
top5_categories = filtered_df.nlargest(5, 'Installs')
top5_categories['Highlight'] = top5_categories['Installs'] > 1000000

top5_categories

In [ ]:
# Define the countries and their weights
countries_data = {
    'Country': [
        'United States',
        'India',
        'China',
        'Brazil',
        'Indonesia',
        'Japan',
        'Germany',
        'United Kingdom',
        'France',
        'Mexico',
        'South Korea',
        'Italy',
        'Russia',
        'Canada',
        'Australia',
        'Spain',
        'Turkey',
        'Saudi Arabia',
        'Nigeria',
        'South Africa',
        'Argentina',
        'Egypt',
        'Pakistan',
        'Vietnam',
        'Thailand',
        'Philippines',
        'Poland',
        'Colombia',
        'Netherlands',
        'Malaysia',
    ],
    'ISO': [
        'USA',
        'IND',
        'CHN',
        'BRA',
        'IDN',
        'JPN',
        'DEU',
        'GBR',
        'FRA',
        'MEX',
        'KOR',
        'ITA',
        'RUS',
        'CAN',
        'AUS',
        'ESP',
        'TUR',
        'SAU',
        'NGA',
        'ZAF',
        'ARG',
        'EGY',
        'PAK',
        'VNM',
        'THA',
        'PHL',
        'POL',
        'COL',
        'NLD',
        'MYS',
    ],
    'Weight': [
        0.17,
        0.13,
        0.11,
        0.065,
        0.05,
        0.045,
        0.04,
        0.035,
        0.03,
        0.03,
        0.025,
        0.025,
        0.02,
        0.02,
        0.02,
        0.018,
        0.015,
        0.015,
        0.015,
        0.012,
        0.01,
        0.01,
        0.01,
        0.008,
        0.007,
        0.006,
        0.006,
        0.005,
        0.005,
        0.004,
    ],
}

countries_df = pd.DataFrame(countries_data)
countries_df['Weight'] = countries_df['Weight'] / countries_df['Weight'].sum()

# Combine Top 5 categories with 30 countries
merged_df = top5_categories.merge(countries_df, how='cross')
merged_df['Country_Installs'] = (merged_df['Installs'] * merged_df['Weight']).astype(int)
merged_df

In [ ]:
# Time Check (6:00 PM to 8:00 PM IST)
ist = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(ist).time()

start_time = datetime.strptime('18:00:00', '%H:%M:%S').time()
end_time = datetime.strptime('20:00:00', '%H:%M:%S').time()

is_within_time = start_time <= current_time < end_time

# Prepare HTML elements
total_global_installs = top5_categories['Installs'].sum()
cat_rows_html = ''
for idx, row in top5_categories.iterrows():
  cat_rows_html += (
      f"<tr><td>{row['Category']}</td><td>{row['Installs']:,}</td></tr>\n"
  )

# Define file paths
graph_path = 'task_2_graph.html'
dashboard_path = 'task_2_dashboard.html'

In [ ]:
# Generating the graph using Plotly and saving it as task_2_graph.html silently if within the allowed time window
if is_within_time:
  categories = top5_categories['Category'].tolist()
  color_scales = ['Viridis', 'Plasma', 'Blues', 'Greens', 'Purples']

  fig = go.Figure()

  for i, cat in enumerate(categories):
    cat_data = merged_df[merged_df['Category'] == cat]
    category_installs = top5_categories.loc[top5_categories['Category'] == cat, 'Installs'].iloc[0]

    fig.add_trace(
        go.Choropleth(
            locations=cat_data['ISO'],
            z=cat_data['Country_Installs'],
            text=cat_data['Country'],
            name='',
            colorscale=color_scales[i % len(color_scales)],
            colorbar_title='Installs',
            visible=(i == 0),
        )
    )

  dropdown_buttons = []

  for i, cat in enumerate(categories):
    visible_flags = [False] * len(categories)
    visible_flags[i] = True

    category_installs = top5_categories.loc[top5_categories['Category'] == cat, 'Installs'].iloc[0]
    label = f'{cat} (>1M)' if category_installs > 1000000 else cat

    dropdown_buttons.append(
        dict(
            label=label,
            method='update',
            args=[
                {'visible': visible_flags},
                {'title.text': f'Global Distributed Installs: <b>{label}</b>'},
            ],
        )
    )

  first_category = categories[0]
  first_installs = top5_categories.loc[top5_categories['Category'] == first_category, 'Installs'].iloc[0]
  first_label = f'{first_category} (>1M)' if first_installs > 1000000 else first_category

  fig.update_layout(
      title=dict(
          text=f'Global Distributed Installs: <b>{first_label}</b>',
          x=0.5,
          y=0.95,
          xanchor='center',
          yanchor='top',
      ),
      updatemenus=[
          dict(
              buttons=dropdown_buttons,
              direction='down',
              showactive=True,
              x=0.0,
              y=1.18,
              xanchor='left',
              yanchor='top',
          )
      ],
      margin=dict(t=100, l=20, r=20, b=20),
      showlegend=False,
  )

  fig.write_html(graph_path)

  status_section_html = f"""
    <div class="card">
        <h2>Graph Status</h2>
        <p style="color: green; font-weight: bold;">Status: Graph Successfully Generated</p>
        <p>The interactive choropleth map has been generated because the execution occurred within the allowed window (6:00 PM - 8:00 PM IST).</p>
        <p><a href="{graph_path}" target="_blank" style="display:inline-block; padding:10px 15px; background-color:#1a73e8; color:white; text-decoration:none; border-radius:5px;">Open Graph</a></p>
    </div>
  """

else:
  status_section_html = f"""
    <div class="card">
        <h2>Graph Status</h2>
        <p style="color: red; font-weight: bold;">Status: Graph Restricted</p>
        <p>The interactive graph is only generated and accessible between <b>6:00 PM IST and 8:00 PM IST</b>. The graph file (<b>{graph_path}</b>) was not created.</p>
    </div>
  """


In [ ]:
# Generating the dashboard HTML content
dashboard_html_content = f"""<!DOCTYPE html>
<html>
<head>
    <title>Play Store Installs Dashboard</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            margin: 30px;
            background-color: #f4f6f9;
            color: #333;
        }}
        h1 {{ color: #1a73e8; }}
        .card {{
            background: white;
            padding: 20px;
            margin-bottom: 20px;
            border-radius: 8px;
            box-shadow: 0 2px 4px rgba(0,0,0,0.1);
        }}
        table {{
            width: 100%;
            border-collapse: collapse;
            margin-top: 10px;
        }}
        th, td {{
            padding: 10px;
            border: 1px solid #ddd;
            text-align: left;
        }}
        th {{ background-color: #f0f4f8; }}
    </style>
</head>
<body>
    <h1>Play Store App Analysis Dashboard</h1>

    <div class="card">
        <h2>Dashboard Summary</h2>
        <p>This dashboard presents global install metrics distributed across <b>30 selected countries</b> based on demographic market weights.</p>
        <p><b>Filter Rules Applied:</b> Excluded categories starting with letters <i>A, C, G, or S</i>.</p>
        <p><b>Highlight Rule:</b> Categories with more than 1,000,000 installs are marked as <b>(>1M)</b> in the dropdown menu.</p>
        <p><b>Total Combined Installs (Top 5):</b> {total_global_installs:,}</p>
    </div>

    <div class="card">
        <h2>Top 5 Categories</h2>
        <table>
            <tr><th>Category Name</th><th>Total Global Installs</th></tr>
            {cat_rows_html}
        </table>
    </div>

    {status_section_html}

</body>
</html>
"""

# Save task_2_dashboard.html
with open(dashboard_path, 'w', encoding='utf-8') as f:
  f.write(dashboard_html_content)